# COIL (2021)
---
[[paper]](https://arxiv.org/pdf/2104.14663)<br>COIL = Contextualized Inverted List

__COIL__ — это метод информационного поиска класса Hybrid Retrieval, цель которого объединить преимущества Dense Retrieval и Sparse Retrieval

__Мотивация__<br>
Dense Retrieval безусловно хорош с точки зрения отлавливания семантики, но имеет свои недостатки: он сложный, эмбеды сжатые (потеря fine-grained информации о отдельных терминах и их роли), неинтерпретируем: не видно где что совпало

Частично эту проблему адресовал **ColBERT (2020)**, за счет использования эмбедингов сразу всех токенов. Это улучшило качество, но всё ещё => было относительно дорогим.

__Идея:__<br>
давайте генерировать описание документа на уровне отдельных токенов как в ColBERT, но вместо агрегации + aNN будем искать документы по старинке - по обратному индексу, перебирая токены запроса. А вот вес мэтча уже по новому через скаолярное произведение сохраняемого в индексе эмбединга

Контекстуализиорваный эмбединг

__Постановка задачи__<br>
По заданному запросу $Q$ найти $K$ наиболее релевантных документов из коллекции $D = \{d_1, d_2, \ldots, d_N\}$

__Альтернативные решения:__<br>
До COIL были и другие походы к Dense Retrieval:
- DPR (2020): два отдельных BERT-энкодера кодируют запроса и документ, которые сравниваются поdot product. contrastive learning (как?)
- ColBERT (2020): перешли на потокенные эмбеды, а вес мэтча - их максимальное произведение
- стек Sparse и Dense Retrieval моделей (сложно и дорого)

DPR не устраивал в качестве (всего один эмбед), ColBERT в скорости (надо поддерживать aNN индекс)

__Архитектура__<br>
BERT-подобная модель генерируют потокенные эмбединги для запроса и докумена. Relevance Score считается как сумма мэтчей, а каждый мэтч - это скалярное произведение

<img src="img/coil.png" width=750>

__Обучение__<br>
Обучали эмбединги с помощью Contrastive Learning, но <u>Supervised</u> его варинта на датасете MS MARCO:
1.  для каждого запроса $Q$, батч состоит из одного позитивного документа $D^+$ (релевантного запросу) и нескольких негативных документов $D^-$ , могут быть "in-batch negatives" (другие позитивные документы из того же батча) или "hard negatives" (найденные с помощью BM25 или других методов, которые похожи на запрос, но нерелевантны).
2.  $Q, D^+, D^-$ пропускаются через общий BERT-энкодер + проекция
5.  для каждого запроса $Q$: $\text{score}(Q, D_k) = \mathbf{v}_Q \cdot \mathbf{v}_{D_k}$.
6.  InfoNCE loss
    $L = -\log \frac{\exp(\text{score}(Q, D^+) / \tau)}{\sum_{D_k \in \{D^+, D^-\}} \exp(\text{score}(Q, D_k) / \tau)}$,
    где $\tau$ — температура, гиперпараметр, регулирующий чувствительность к негативным примерам. Цель — максимизировать скор для позитивных пар и минимизировать для негативных.
7.  обучаем методом обратного распространения ошибки для минимизации функции потерь

__Построение индекса__<br>
1.  каждый документ $D_k$ из коллекции пропускается через обученный BERT-энкодер
2.  создаем Inverted Index: если токен есть в документе, добавляем id документа и эмбединг токена внутри документа в posting list данного токена

__Инференс__<br>
1.  запрос $Q$ пропускается через обученный COIL энкодер проекцией в сжатое 32 предстваление
2.  для каждого токена $q_i$ запроса $Q$ ищем документы по инвертированному индексу
    -  для каждого документа из этого индекса считаем скларяное произведение с ним $<\mathbf{v}_i, \mathbf{v}_j>$
3. сумиируем по документу - это будет relevance score
4. ранжируем документы по убыванию relevance score

__Результаты__<br>
На датасете MS MARCO модель COIL:
- точнее BM25 на 15–20 пунктов по метрике MRR@10<br>MRR - на какой позиции в выдаче правильный ответ
- значительно быстрее ColBERT из-за поиска сразу в инверированном индексе
- как и ColBERT хорошо учитывает новые (OOV) термины, поскольку учитывает более детальный контекст (на уровне токенов)